In [2]:
! pip install -q datasets==2.21.0 requests torch peft bitsandbytes transformers==4.43.1 trl accelerate sentencepiece


In [3]:
import os
import re
import math
from tqdm import tqdm
from huggingface_hub import login
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, Trainer,set_seed
from peft import LoraConfig,PeftModel
from datetime import datetime


c:\Users\KRISH\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
BASE_MODEL="meta-llama/Meta-Llama-3.1-8B"
FINETUNED_MODEL=f"ed-donner/pricer-2024-09-13_13.04.39"
# 

#  HYPERPARAMETERS 
LORA_R=32
LORA_ALPHA=64
TARGET_MODULES=["q_proj", "v_proj", "k_proj", "o_proj"]


## Login to Hugging Face

In [10]:

hf_token = os.getenv("HF_TOKEN")


In [ ]:
base_model= AutoModelForCausalLM.from_pretrained(BASE_MODEL, device_map="auto")

In [12]:
print(f"Memory footprint of the base model: {base_model.get_memory_footprint()/1e9:,.1f} GB")

NameError: name 'base_model' is not defined

In [ ]:
base_model

In [ ]:

quant_config=BitsAndBytesConfig(load_in_8bit=True)
base_model=AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=quant_config, device_map="auto")


In [ ]:
print(f"Memory footprint of the base model: {base_model.get_memory_footprint()/1e9:,.1f} GB")


In [ ]:
quant_config=BitsAndBytesConfig(load_in_4bit=True,
                                bnb_4bit_use_double_quant=True, ## squeezes the value reduces memory sapace
                                bnb_4bit_compute_dtype=torch.bfloat16,
bnb_4bit_quant_type="nf4" # interpret the 4 bit number as a normal float, not a int
)

base_model=AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, 
        quantization_config=quant_config, device_map="auto")

In [ ]:
print(f"Memory footprint of the base model: {base_model.get_memory_footprint()/1e9:,.1f} GB")

In [ ]:
fine_tuned_model = PeftModel.from_pretrained(base_model, FINETUNED_MODEL, device_map="auto")

In [ ]:
# print(f"Memory footprint of the base model: {base_model.get_memory_footprint()/1e9:,.1f} GB")
print(f"Memory footprint of the fine-tuned model: {fine_tuned_model.get_memory_footprint()/1e9:,.1f} GB")

In [ ]:
fine_tuned_model

In [16]:
# Each of the Target Modules has 2 LoRA Adaptor matrices, called Lora_A and lora_B
# These are designed so that weights can be adapted by adding alpha*lora_a*lora_b

lora_q_proj= 4096*32 + 4096*32
lora_k_proj= 4096*32 + 1024*32
lora_v_proj= 4096*32 + 1024*32
lora_o_proj= 4096*32 + 4096*32

lora_layer = lora_q_proj + lora_k_proj + lora_v_proj + lora_o_proj
params=lora_layer * 32
size=params * 4 / 1e6 # size in MB
print(f"Size of the LoRA Adapters: {size:.2f} MB and parameters: {params:,}")

Size of the LoRA Adapters: 109.05 MB and parameters: 27,262,976
